# `ModelRetryMiddleware`

Middleware that automatically retries failed model calls using configurable exception filtering and backoff delays.

It intercepts model execution, retries matching exceptions, and decides what to do when all retry attempts are exhausted.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
ModelRetryMiddleware(
    *,
    max_retries: int = 2, # Retries after the initial model call
    retry_on: RetryOn = (Exception,), # Exceptions that trigger a retry
    on_failure: OnFailure = "continue", # Behaviour after retries are exhausted
    backoff_factor: float = 2.0, # Exponential backoff multiplier
    initial_delay: float = 1.0, # Delay before the first retry
    max_delay: float = 60.0, # Maximum delay between retries
    jitter: bool = True # Adds ±25% random delay variation
)
```

## Parameters

* `max_retries` — Maximum number of retry attempts after the initial model call.
  * Default: `2`
  * Must be greater than or equal to `0`.
  * Total possible attempts are `max_retries + 1`.

* `retry_on` — Determines which exceptions should trigger another attempt.
  * Default: `(Exception,)`, which retries all standard exceptions.
  * May be a tuple of exception classes.
  * May be a callable that receives an exception and returns `True` or `False`.

* `on_failure` — Determines what happens when an exception is not retryable or all retries are exhausted.
  * `"continue"` — Returns a `ModelResponse` containing an error `AIMessage`.
  * `"error"` — Re-raises the original exception.
  * Callable — Receives the final exception and returns custom text for an `AIMessage`.

* `backoff_factor` — Multiplier used to increase the delay between retries.
  * Default: `2.0`
  * Must be greater than or equal to `0`.
  * Set it to `0.0` to use a constant delay.

* `initial_delay` — Delay in seconds before the first retry.
  * Default: `1.0`
  * Must be greater than or equal to `0`.

* `max_delay` — Maximum delay in seconds between attempts.
  * Default: `60.0`
  * Must be greater than or equal to `0`.
  * Caps exponential delay growth.

* `jitter` — Whether to add random variation of up to `±25%` to each delay.
  * Default: `True`
  * Helps prevent many requests from retrying simultaneously.

## Type Aliases

### `RetryOn`

Defines which exceptions are retryable.

```python
RetryOn = (
    tuple[type[Exception], ...]
    | Callable[[Exception], bool]
)
```

It may contain:

* A tuple of exception classes checked through `isinstance`.
* A callable that returns `True` when an exception should be retried.

### `OnFailure`

Defines how the middleware handles a final failure.

```python
OnFailure = (
    Literal["error", "continue"]
    | Callable[[Exception], str]
)
```

Supported values:

* `"continue"` — Converts the failure into an `AIMessage`.
* `"error"` — Re-raises the exception.
* Callable — Produces custom error-message content.

## Attributes

* `max_retries` — Number of retries allowed after the initial call.
* `retry_on` — Exception tuple or exception-filtering callable.
* `on_failure` — Final failure-handling strategy.
* `backoff_factor` — Exponential backoff multiplier.
* `initial_delay` — Delay before the first retry.
* `max_delay` — Maximum permitted retry delay.
* `jitter` — Whether random delay variation is enabled.
* `tools` — Empty list because this middleware does not register agent tools.

## Methods

1. `_format_failure_message`: Creates the default error `AIMessage`.
   * Includes the number of attempts made.
   * Includes the exception class name.
   * Includes the exception message.
   - **Syntax:**
     ```python
     _format_failure_message(
         exc: Exception, # Exception that caused the failure
         attempts_made: int # Number of completed attempts
     ) -> AIMessage
     ```

2. `_handle_failure`: Applies the configured final failure behaviour.
   * Re-raises the exception when `on_failure="error"`.
   * Uses a custom callable when one was supplied.
   * Otherwise returns the default formatted error response.
   - **Syntax:**
     ```python
     _handle_failure(
         self,
         exc: Exception, # Final or non-retryable exception
         attempts_made: int # Number of attempts already made
     ) -> ModelResponse[ResponseT]
     ```

3. `wrap_model_call`: Executes a synchronous model call with retry logic.
   * Makes the initial model call.
   * Checks whether a caught exception is retryable.
   * Waits using `time.sleep` before each retry.
   * Returns immediately when an attempt succeeds.
   * Applies `on_failure` when no further retry should occur.
   - **Syntax:**
     ```python
     wrap_model_call(
         self,
         request: ModelRequest[ContextT], # Model request
         handler: Callable[
             [ModelRequest[ContextT]],
             ModelResponse[ResponseT]
         ] # Function that executes the request
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

4. `awrap_model_call`: Asynchronous version of `wrap_model_call`.
   * Uses the same retry and failure-handling rules.
   * Awaits the model handler.
   * Uses `asyncio.sleep` instead of blocking execution.
   - **Syntax:**
     ```python
     async def awrap_model_call(
         self,
         request: ModelRequest[ContextT], # Model request
         handler: Callable[
             [ModelRequest[ContextT]],
             Awaitable[ModelResponse[ResponseT]]
         ] # Async function that executes the request
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

## Retry Flow

```text
Make the initial model call
        |
        v
Did the call succeed? -------- Yes -------> Return the response
        |
        No
        |
        v
Is the exception retryable? -- No --------> Apply on_failure
        |
        Yes
        |
        v
Are retries remaining? ------- No --------> Apply on_failure
        |
        Yes
        |
        v
Calculate delay -> Wait -> Try the model call again
```

A non-retryable exception is handled immediately, even when unused retry attempts remain.

## Backoff Calculation

When `backoff_factor` is not `0.0`, the base delay is calculated as:

```python
delay = initial_delay * (backoff_factor ** retry_number)
```

Here, `retry_number` is zero-based:

```text
Before retry 1: retry_number = 0
Before retry 2: retry_number = 1
Before retry 3: retry_number = 2
```

The calculated value is capped at `max_delay`.

### Example

```python
ModelRetryMiddleware(
    max_retries=4,
    initial_delay=1.0,
    backoff_factor=2.0,
    max_delay=60.0,
    jitter=False
)
```

The delays are:

```text
Before retry 1: 1 × 2⁰ = 1 second
Before retry 2: 1 × 2¹ = 2 seconds
Before retry 3: 1 × 2² = 4 seconds
Before retry 4: 1 × 2³ = 8 seconds
```

### Constant Delay

Set `backoff_factor=0.0` to keep the delay constant:

```python
ModelRetryMiddleware(
    max_retries=3,
    initial_delay=2.0,
    backoff_factor=0.0,
    jitter=False
)
```

```text
Before retry 1: 2 seconds
Before retry 2: 2 seconds
Before retry 3: 2 seconds
```

### Jitter

When `jitter=True`, the middleware randomly adjusts the capped delay by up to `±25%`.

For a base delay of `4` seconds, the actual delay may be between approximately:

```text
3 seconds and 5 seconds
```

The final delay is never allowed to become negative.

## Failure Handling

### Continue Agent Execution

```python
retry = ModelRetryMiddleware(
    max_retries=2,
    on_failure="continue"
)
```

After the final failure, the middleware returns a `ModelResponse` containing an `AIMessage` similar to:

```text
Model call failed after 3 attempts with APITimeoutError: Request timed out
```

### Re-raise the Exception

```python
retry = ModelRetryMiddleware(
    max_retries=2,
    on_failure="error"
)
```

The final exception is raised and agent execution stops.

### Custom Failure Message

```python
def format_error(exc: Exception) -> str:
    return "The model is temporarily unavailable. Please try again later."

retry = ModelRetryMiddleware(
    max_retries=3,
    on_failure=format_error
)
```

The callable's returned string becomes the content of the error `AIMessage`.

## Exceptions

The constructor raises `ValueError` when:

```text
max_retries < 0
initial_delay < 0
max_delay < 0
backoff_factor < 0
```

The sync and async retry methods contain a defensive `RuntimeError` for the unreachable case where the retry loop completes without returning.

## Examples

### Basic Usage

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[search_tool],
    middleware=[ModelRetryMiddleware()]
)
```

This allows two retries after the initial model call.

### Retry Specific Exceptions

```python
from anthropic import RateLimitError
from openai import APITimeoutError

retry = ModelRetryMiddleware(
    max_retries=4,
    retry_on=(APITimeoutError, RateLimitError),
    backoff_factor=1.5
)
```

Other exception types are handled immediately according to `on_failure`.

### Use a Custom Exception Filter

```python
from anthropic import APIStatusError

def should_retry(exc: Exception) -> bool:
    if isinstance(exc, APIStatusError):
        return 500 <= exc.status_code < 600
    return False

retry = ModelRetryMiddleware(
    max_retries=3,
    retry_on=should_retry
)
```

Only server-side HTTP errors from `500` through `599` are retried.

### Disable Waiting

```python
retry = ModelRetryMiddleware(
    max_retries=3,
    initial_delay=0.0
)
```

The middleware retries immediately without sleeping.

## Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/model_retry.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```